# Reasoner Notebook — PELLET

Notebook này chạy **Pellet Reasoner** trên file `v-idiomV5.rdf`.

| | Pellet | HermiT |
|---|---|---|
| **Tốc độ** | Nhanh hơn (tối ưu cho dữ liệu thực thể) | Chậm hơn |
| **Độ chính xác** | Tốt với ABox (individual instances) | Mạnh hơn với TBox phức tạp |
| **Yêu cầu** | Java | Java |

> **⚠️ File `v-idiomV5.rdf` gốc sẽ KHÔNG bị thay đổi.**

In [1]:
# CELL 1: TẢI ONTOLOGY
from owlready2 import *
BASE_IRI = "http://www.semanticweb.org/phandangvu/ontologies/2026/8/v-idiomv2#"

temp_world = World()
onto_r = temp_world.get_ontology("../../ontology_protege/v-idiomV5_final.rdf").load()
print("✅ Đã tải:", onto_r.base_iri)

✅ Đã tải: http://www.semanticweb.org/phandangvu/ontologies/2026/8/v-idiomv2#


In [2]:
# CELL 2: CHẠY REASONER + ĐO THỜI GIAN
import time

print("🟣 Đang chạy Reasoner...")
t_start = time.time()

with onto_r:
    # Tham số cực kỳ quan trọng để Reasoner nạp thuộc tính suy luận (Có_nghĩa) vào object
    sync_reasoner(x=temp_world, infer_property_values=True)   

t_elapsed = time.time() - t_start
print(f"✅ Reasoner chạy xong! Thời gian: {t_elapsed:.3f} giây")


🟣 Đang chạy Reasoner...


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/hermit:/Users/phandangvu/Desktop/ontology/onto-python/.venv/lib/python3.12/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/49/lh523xrs2ds76mm2n3922kww0000gn/T/tmp09tp8vqf -Y


✅ Reasoner chạy xong! Thời gian: 0.614 giây


* Owlready2 * HermiT took 0.5929038524627686 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [3]:
# CELL 3: IN KẾT QUẢ ĐỒNG NGHĨA DỰA 100% VÀO THUỘC TÍNH SUY LUẬN "CÓ_NGHĨA"
Khung_cls = temp_world[BASE_IRI + "Khung_Bản_Thể_học"]
VN_cls    = temp_world[BASE_IRI + "Vietnamese_idiom"]
EN_cls    = temp_world[BASE_IRI + "English_Idiom"]

def get_label(entity):
    if entity and hasattr(entity, "comment") and entity.comment:
        return f"{entity.name} ({entity.comment[0]})"
    return entity.name if entity else ""

print("=" * 80)
print("  [REASONER] TÌM ĐỒNG NGHĨA BẰNG PROPERTY CHAIN 'CÓ_NGHĨA'")
print("=" * 80)

seen_frames = set() # Chống in trùng nhóm
count = 0

for vi in VN_cls.instances():
    # Sức mạnh của Reasoner: Thuộc tính Có_nghĩa tự động sinh ra và có sẵn
    # Lấy ngay danh sách các câu tiếng Anh nằm trong danh sách đồng nghĩa của câu này
    synonyms_en = [inst for inst in vi.Có_nghĩa if type(inst) is EN_cls]
    
    if synonyms_en:
        khung = vi.Có_khung[0] if vi.Có_khung else None
        if khung and khung.name not in seen_frames:
            seen_frames.add(khung.name)
            count += 1
            
            print(f"\n📦 {khung.name}")
            if khung.Có_bối_cảnh:  print(f"   Bối cảnh : {get_label(khung.Có_bối_cảnh[0])}")
            if khung.Có_hành_động: print(f"   Hành động: {get_label(khung.Có_hành_động[0])}")
            if khung.Có_kết_quả:   print(f"   Kết quả  : {get_label(khung.Có_kết_quả[0])}")
            if khung.Có_mục_đích:  print(f"   Mục đích : {get_label(khung.Có_mục_đích[0])}")
            print()
            
            # Lấy tất cả câu VN trong chuỗi đồng nghĩa
            all_vn = set([vi] + [inst for inst in vi.Có_nghĩa if type(inst) is VN_cls])
            
            for v in all_vn:
                meaning = v.comment[0] if v.comment else ""
                print(f"   🇻🇳  {v.name.replace('_', ' ')}")
                if meaning: print(f"       => {meaning}")
                
            for e in synonyms_en:
                meaning = e.comment[0] if e.comment else ""
                print(f"   🇬🇧  {e.name.replace('_', ' ')}")
                if meaning: print(f"       => {meaning}")
                
            print("-" * 80)

print(f"\n✅ Tổng cộng: {count} nhóm đồng nghĩa (Truy xuất thuần túy bằng thuộc tính suy luận Có_nghĩa).")


  [REASONER] TÌM ĐỒNG NGHĨA BẰNG PROPERTY CHAIN 'CÓ_NGHĨA'

📦 Khung1
   Bối cảnh : BC_Đối_nhân_xử_thế (Ứng xử xã hội giữa người với người)
   Hành động: HD_Lợi_dụng (Lợi dụng, vô ơn, ăn cháo đá bát)
   Kết quả  : KQ_Mất_quan_hệ (Mất quan hệ, gây thù chuốc oán)

   🇻🇳  Ăn cháo đá bát
       => Phản bội, đối xử tệ bạc với người đã cưu mang, giúp đỡ mình.
   🇻🇳  Có mới nới cũ
       => Có cái mới, bạn mới thì bỏ rơi, lạnh nhạt với cái cũ, người cũ.
   🇬🇧  Bite the hand that feeds one
       => To treat a benefactor with ingratitude or hostility after receiving help.
   🇬🇧  Out with the old in with the new
       => To abandon or neglect old friends or items as soon as new ones appear.
--------------------------------------------------------------------------------

📦 Khung2
   Bối cảnh : BC_Cảm_xúc (Cảm xúc, tâm lý, nội tâm)
   Hành động: HD_So_sánh (So sánh, nhìn sang hoàn cảnh người khác)

   🇻🇳  Đứng núi này trông núi nọ
       => Bất mãn với hoàn cảnh hiện tại, luôn nghĩ hoàn cảnh ngư